# SwinIR Super-Resolution Inference

This notebook runs the repository's patched **SwinIR** inference script on
a directory of low-resolution (LR) tiles and writes the super-resolved
outputs to the SwinIR `results/` directory.

The workflow is based on the existing `main_test_swinir.py` used in the
original SwinIR notebook. In this repository, that script has been adapted
to support more flexible filename matching for paired datasets.

## What this notebook does

- Uses the existing `main_test_swinir.py` script already present in the repo.
- Runs 4x real-world super-resolution on a folder of LR tiles.
- Supports both official pre-trained checkpoints and exported fine-tuned
  checkpoints.
- Documents a common checkpoint / architecture mismatch issue and how to fix it.

## What this notebook does *not* do

- **Training / fine-tuning** — see `SwinIR_Training.ipynb`.
- **Evaluation (PSNR / SSIM)** — see `SwinIR_Evaluation.ipynb`.
- **Tile stitching / map reconstruction** — run your mapping notebook separately.

## Prerequisites

1. The SwinIR repository exists locally.
2. The repository contains the patched `main_test_swinir.py`.
3. Supporting files are available in the expected repo locations:
   - `models/network_swinir.py`
   - `utils/util_calculate_psnr_ssim.py`
4. A compatible SwinIR checkpoint is available.
5. A folder of LR input tiles already exists.
6. PyTorch can access a CUDA device if GPU inference is desired.

## Important note on outputs

`main_test_swinir.py` saves predictions as `<image_id>_SwinIR.png` inside a
results folder such as `results/swinir_real_sr_x4_large/`. The evaluation and
visualization scripts in this repo already handle that naming convention.



In [ ]:
# Imports
import shlex
import subprocess
from pathlib import Path

In [ ]:
# Configuration
# Adjust these paths to your local setup.

REPO_ROOT = Path('/home/datalab/Datalab/swinir_repo') # Change this to your local SwinIR repository path.
INFER_SCRIPT = REPO_ROOT / 'main_test_swinir.py' # Path to the inference script.
NETWORK_FILE = REPO_ROOT / 'models' / 'network_swinir.py' # Path to the network definition file.
UTIL_FILE = REPO_ROOT / 'utils' / 'util_calculate_psnr_ssim.py' # Path to the PSNR/SSIM utility file.

# Input LR tiles.
INPUT_DIR = Path('/home/datalab/Datalab/data/inputs') # Change this to your local input images path.

# Official pre-trained large-model checkpoint for real-world SR x4.
MODEL_PATH = REPO_ROOT / 'model_zoo' / 'swinir' / '003_realSR_BSRGAN_DFOWMFC_s64w8_SwinIR-L_x4_GAN.pth' # Change this to your local model checkpoint path.

TASK = 'real_sr' # Task type: 'classical_sr', 'lightweight_sr', 'real_sr', 'gray_dn', 'color_dn', 'jpeg_car'
SCALE = 4
USE_LARGE_MODEL = True
TILE = 400               # Reduce further if GPU memory is limited.
TILE_OVERLAP = 32

# The official script writes to results/ automatically.
RESULTS_ROOT = REPO_ROOT / 'results'
EXPECTED_RESULT_DIR = RESULTS_ROOT / (
    'swinir_real_sr_x4_large' if USE_LARGE_MODEL else 'swinir_real_sr_x4') # Change this to your local expected results path.

print(f'Repo root : {REPO_ROOT}')
print(f'Input dir : {INPUT_DIR}')
print(f'Model path: {MODEL_PATH}')
print(f'Results   : {EXPECTED_RESULT_DIR}')



## Repository sanity check

Confirm that the required files exist before running inference.



In [ ]:
required_files = [INFER_SCRIPT, NETWORK_FILE, UTIL_FILE]
missing_files = [path for path in required_files if not path.exists()]

if missing_files:
    raise FileNotFoundError(
        'Missing required files:\n' + '\n'.join(str(path) for path in missing_files))

print('Required inference files found.')



## Build and run the command

The helper below assembles the inference command in a readable way, then
runs it from the repository root.



In [ ]:
def build_inference_command(
    script_path: Path,
    model_path: Path,
    folder_lq: Path,
    task: str = 'real_sr',
    scale: int = 4,
    large_model: bool = False,
    tile: int | None = None,
    tile_overlap: int = 32,
) -> list[str]:
    """Build the SwinIR inference command."""
    cmd = [
        'python',
        str(script_path),
        '--task', task,
        '--scale', str(scale),
        '--model_path', str(model_path),
        '--folder_lq', str(folder_lq)]

    if large_model:
        cmd.append('--large_model')
    if tile is not None:
        cmd += ['--tile', str(tile)]
    if tile_overlap is not None:
        cmd += ['--tile_overlap', str(tile_overlap)]
    return cmd


def run_inference(cmd: list[str], repo_root: Path) -> None:
    """Execute the SwinIR inference script."""
    print('Running command:')
    print(' '.join(shlex.quote(part) for part in cmd))
    subprocess.run(cmd, cwd=repo_root, check=True)


cmd = build_inference_command(
    script_path=INFER_SCRIPT,
    model_path=MODEL_PATH,
    folder_lq=INPUT_DIR,
    task=TASK,
    scale=SCALE,
    large_model=USE_LARGE_MODEL,
    tile=TILE,
    tile_overlap=TILE_OVERLAP)

run_inference(cmd, REPO_ROOT)



## Common checkpoint mismatch issue

A common error in real-SR inference is a mismatch between:

- the **checkpoint architecture**, and
- the `--large_model` flag.

In practice, the two most common configurations are:

- **SwinIR-M x4 GAN** — medium model, do **not** pass `--large_model`.
- **SwinIR-L x4 GAN** — large model, **do** pass `--large_model`.

If the configuration does not match the checkpoint, you may see a loading
error with many missing keys or an unexpected `params_ema` entry.

The original notebook already showed this behavior: the medium-model command
failed, while the large-model command succeeded with the matching official
checkpoint.



## Optional: run a fine-tuned checkpoint

After fine-tuning, export an inference-ready checkpoint such as
`best_inference.pth` and point `MODEL_PATH` to that file. If the model was
trained from the large SwinIR-L backbone, keep `USE_LARGE_MODEL=True`.



In [ ]:
# Example configuration for a fine-tuned checkpoint.
# Uncomment and run this cell when you want to use your own weights.

# FINETUNED_MODEL_PATH = REPO_ROOT / 'experiments' / 'swinir_finetune_iter_1' / 'checkpoints' / 'best_inference.pth' # Change this to your local fine-tuned model checkpoint path.
# TEST_INPUT_DIR = Path('/home/datalab/zDatalab2/dataset/test/inputs') # Change this to your local test input images path.
#
# finetuned_cmd = build_inference_command(
#     script_path=INFER_SCRIPT,
#     model_path=FINETUNED_MODEL_PATH,
#     folder_lq=TEST_INPUT_DIR,
#     task=TASK,
#     scale=SCALE,
#     large_model=True,
#     tile=TILE,
#     tile_overlap=TILE_OVERLAP)
#
# run_inference(finetuned_cmd, REPO_ROOT)



## Notes on the patched script

In this repository, `main_test_swinir.py` includes a modification that makes
paired filename matching more flexible for custom datasets. For classical SR
and lightweight SR tasks, it can now try multiple LR filename patterns instead
of requiring the original SwinIR naming convention only.

This change is especially useful when LR and GT images share the same stem but
use different extensions or do not include the original `x4` suffix.



## Notes on outputs

The inference script creates an output subdirectory inside `results/`.
For the large real-SR model, the default folder is:

- `results/swinir_real_sr_x4_large/`

These SR tiles can then be:

- evaluated in `SwinIR_Evaluation.ipynb`, or
- stitched back into a full map with your mapping notebook.

If you run out of GPU memory, reduce `TILE` further.



---
